In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Load historical stock prices
prices = pd.read_csv(
    "five_stock_prices.csv",
    index_col=0,
    parse_dates=True
)

prices = prices.sort_index().dropna()

# Calculate daily returns
returns = prices.pct_change().dropna()

# Chronological 70% training / 30% testing split
split = int(len(returns) * 0.70)

train = returns.iloc[:split]
test = returns.iloc[split:]

print("Stocks:", list(returns.columns))
print("Training period:", train.index.min().date(),
      "to", train.index.max().date())
print("Testing period:", test.index.min().date(),
      "to", test.index.max().date())
print("Training days:", len(train))
print("Testing days:", len(test))

assert len(train) > 0 and len(test) > 0
assert train.index.max() < test.index.min()

Stocks: ['AAPL', 'BLK', 'GS', 'JPM', 'MSFT']
Training period: 2025-09-24 to 2026-06-03
Testing period: 2026-06-04 to 2026-09-21
Training days: 174
Testing days: 75


In [2]:


# Estimate annualized covariance using training data only
cov = train.cov().values * 252
assets = list(train.columns)
n = len(assets)

# 1. Equal-weight portfolio
equal_weights = np.ones(n) / n

# 2. Minimum-variance portfolio
def portfolio_volatility(w):
    return np.sqrt(w @ cov @ w)

constraints = {"type": "eq", "fun": lambda w: w.sum() - 1}
bounds = [(0, 1)] * n

min_var_result = minimize(
    portfolio_volatility,
    equal_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 2000, "ftol": 1e-12}
)

assert min_var_result.success, min_var_result.message
min_var_weights = min_var_result.x

# 3. Risk-parity portfolio
def risk_contributions(w):
    vol = portfolio_volatility(w)
    return w * (cov @ w) / vol

def risk_parity_objective(w):
    contributions = risk_contributions(w)
    target = contributions.sum() / n
    return np.sum((contributions - target) ** 2)

rp_result = minimize(
    risk_parity_objective,
    equal_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 5000, "ftol": 1e-14}
)

assert rp_result.success, rp_result.message
rp_weights = rp_result.x

# Verify that the risk contributions are approximately equal
rp_contributions = risk_contributions(rp_weights)
assert np.allclose(
    rp_contributions / rp_contributions.sum(),
    np.ones(n) / n,
    atol=0.01
)

weights = pd.DataFrame({
    "Equal Weight (%)": equal_weights * 100,
    "Minimum Variance (%)": min_var_weights * 100,
    "Risk Parity (%)": rp_weights * 100
}, index=assets)

display(weights.round(2))

print("\nTraining-period annualized volatility:")
for name, w in [
    ("Equal Weight", equal_weights),
    ("Minimum Variance", min_var_weights),
    ("Risk Parity", rp_weights)
]:
    print(f"{name}: {portfolio_volatility(w):.2%}")

,Equal Weight (%),Minimum Variance (%),Risk Parity (%)
AAPL,20.0,37.38,25.00
BLK,20.0,9.02,17.17
GS,20.0,0.00,14.19
JPM,20.0,28.63,20.78
MSFT,20.0,24.96,22.86



Training-period annualized volatility:
Equal Weight: 17.67%
Minimum Variance: 15.71%
Risk Parity: 16.85%
